In [ ]:
!pip install -q huggingface_hub transformers torch shap datasets pandas

In [ ]:
import shap
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
!pip install --upgrade huggingface_hub transformers # `huggingface_hub` 및 `transformers` 버전 충돌 해결
from transformers import AutoModel, AutoTokenizer, ElectraModel, AutoModelForCausalLM, pipeline
from huggingface_hub import hf_hub_download

In [ ]:
url = "https://huggingface.co/datasets/AKS-DHLAB/KPoEM/resolve/main/KPoEM_line_dataset_v4.tsv"

df = pd.read_csv(
    url,
    sep="\t",
    encoding="utf-8",
    quoting=3
)

df.head()


,line_id,poem_id,text,sub_title,title,poet,annotator_01,annotator_02,annotator_03,annotator_04,annotator_05
0,1,1,죽는 날까지 하늘을 우러러,NaN,서시,윤동주,비장함,비장함,"뿌듯함, 비장함","비장함, 뿌듯함, 감동/감탄","비장함, 서러움, 슬픔"
1,2,1,"한 점 부끄럼이 없기를,",NaN,서시,윤동주,"부끄러움, 비장함","부끄러움, 비장함, 기대감, 불안/걱정, 서러움, 슬픔","깨달음, 비장함, 뿌듯함","비장함, 부끄러움, 기대감",비장함
2,3,1,잎새에 이는 바람에도,NaN,서시,윤동주,"기대감, 신기함/관심","기대감, 불안/걱정, 비장함","슬픔, 서러움, 불안/걱정, 당황/난처","비장함, 슬픔","감동/감탄, 신기함/관심, 편안/쾌적, 기대감"
3,4,1,나는 괴로워했다.,NaN,서시,윤동주,"절망, 슬픔, 패배/자기혐오","절망, 슬픔, 패배/자기혐오, 죄책감, 힘듦/지침, 비장함","당황/난처, 서러움, 죄책감, 패배/자기혐오","비장함, 슬픔, 패배/자기혐오, 절망, 힘듦/지침","슬픔, 서러움, 절망, 힘듦/지침, 패배/자기혐오"
4,5,1,별을 노래하는 마음으로,NaN,서시,윤동주,"기쁨, 신기함/관심, 즐거움/신남, 흐뭇함(귀여움/예쁨), 뿌듯함","기쁨, 뿌듯함, 감동/감탄, 슬픔, 비장함, 아껴주는, 환영/호의, 기대감","고마움, 기대감, 기쁨, 아껴주는, 흐뭇함(귀여움/예쁨)","감동/감탄, 기대감, 기쁨, 아껴주는, 행복","즐거움/신남, 기대감, 기쁨, 행복"


In [ ]:
url = "https://huggingface.co/datasets/AKS-DHLAB/KPoEM/resolve/main/KPoEM_poem_dataset_v4.tsv"

df_poem = pd.read_csv(
    url,
    sep="\t",
    encoding="utf-8",
    quoting=3
)

df_poem.head(10)


,seg_id,poem_id,text,sub_title,title,poetry_book,poet,annotator_01,annotator_02,annotator_03,annotator_04,annotator_05
0,1,1,"죽는 날까지 하늘을 우러러 한 점 부끄럼이 없기를, 잎새에 이는 바람에도 나는 괴로...",NaN,서시,하늘과 바람과 별과 시,윤동주,"불안/걱정, 비장함, 서러움, 슬픔, 안타까움/실망, 부끄러움, 죄책감, 패배/자기혐오","패배/자기혐오, 서러움, 비장함, 부끄러움, 아껴주는, 슬픔, 불안/걱정, 죄책감","깨달음, 흐뭇함(귀여움/예쁨), 고마움, 힘듦/지침, 안타까움/실망","비장함, 뿌듯함, 슬픔","비장함, 감동/감탄, 깨달음, 서러움"
1,2,2,산모퉁이를 돌아 논가 외딴 우물을 홀로 찾아가선 가만히 들여다봅니다. 우물 속에는 ...,NaN,자화상,하늘과 바람과 별과 시,윤동주,"불쌍함/연민, 아껴주는, 신기함/관심, 흐뭇함(귀여움/예쁨), 서러움, 짜증, 지긋...","신기함/관심, 증오/혐오, 불쌍함/연민, 화남/분노, 서러움, 슬픔, 아껴주는, 패...","안타까움/실망, 서러움, 슬픔, 깨달음, 흐뭇함(귀여움/예쁨)","불평/불만, 안타까움/실망, 의심/불신, 슬픔","불쌍함/연민, 불안/걱정"
2,3,3,쫓아오든 햇빛인데 지금 교회당 꼭대기 십자가에 걸리었습니다. 첨탑(尖塔)이 저렇게...,NaN,십자가,하늘과 바람과 별과 시,윤동주,"서러움, 패배/자기혐오, 존경, 감동/감탄, 비장함, 불쌍함/연민, 불안/걱정, 죄책감","감동/감탄, 놀람, 불쌍함/연민, 비장함, 슬픔, 행복, 아껴주는, 불안/걱정","절망, 깨달음, 서러움, 힘듦/지침, 존경, 비장함","당황/난처, 힘듦/지침, 놀람, 슬픔, 불안/걱정","깨달음, 비장함, 존경, 신기함/관심"
3,4,4,바람이 어디로부터 불어 와 어디로 불려 가는 것일까 바람이 부는데 내 괴로움에는 ...,NaN,바람이 불어,하늘과 바람과 별과 시,윤동주,"슬픔, 서러움, 한심함, 패배/자기혐오, 죄책감, 부끄러움","비장함, 서러움, 슬픔, 불안/걱정, 패배/자기혐오, 죄책감","힘듦/지침, 안타까움/실망, 서러움, 절망, 깨달음","힘듦/지침, 당황/난처, 불안/걱정, 불평/불만, 슬픔, 부끄러움, 패배/자기혐오","깨달음, 불쌍함/연민, 불안/걱정, 서러움, 패배/자기혐오, 힘듦/지침"
4,5,5,고향에 돌아온 날 밤에 내 백골(白骨)이 따라와 한방에 누웠다. 어둔 방은 우주로...,NaN,또 다른 고향,하늘과 바람과 별과 시,윤동주,"불쌍함/연민, 비장함, 공포/무서움, 슬픔, 서러움, 의심/불신, 깨달음, 기대감","힘듦/지침, 놀람, 서러움, 슬픔, 감동/감탄, 존경, 죄책감, 패배/자기혐오, 비...","절망, 깨달음, 서러움, 힘듦/지침, 안타까움/실망","슬픔, 힘듦/지침, 절망, 기대감, 안타까움/실망, 불안/걱정","공포/무서움, 당황/난처, 놀람, 불안/걱정, 비장함"
5,6,6,계절이 지나가는 하늘에는 가을로 가득 차 있습니다. 나는 아무 걱정도 없이 가을 ...,NaN,별 헤는 밤,하늘과 바람과 별과 시,윤동주,"흐뭇함(귀여움/예쁨), 감동/감탄, 아껴주는, 슬픔, 기쁨, 기대감, 깨달음, 환영...","감동/감탄, 기대감, 서러움, 슬픔, 비장함, 아껴주는, 흐뭇함(귀여움/예쁨), 안...","흐뭇함(귀여움/예쁨), 기대감, 안타까움/실망, 서러움, 슬픔","감동/감탄, 기쁨, 흐뭇함(귀여움/예쁨), 행복, 아껴주는, 편안/쾌적","존경, 감동/감탄, 신기함/관심, 깨달음, 서러움, 슬픔, 안타까움/실망"
6,7,6,"어머님, 나는 별 하나에 아름다운 말 한 마디씩 불러 봅니다. 소학교 때 책상을 같...",NaN,별 헤는 밤,하늘과 바람과 별과 시,윤동주,"아껴주는, 슬픔, 부끄러움, 불쌍함/연민, 존경, 감동/감탄, 고마움, 흐뭇함(귀여...","아껴주는, 흐뭇함(귀여움/예쁨), 환영/호의, 서러움, 슬픔, 깨달음, 죄책감, 부...","안타까움/실망, 서러움, 기대감, 슬픔, 깨달음","불안/걱정, 서러움, 슬픔, 존경, 환영/호의","서러움, 슬픔, 존경"
7,8,7,"살구나무 그늘로 얼굴을 가리고, 병원 뒤뜰에 누워, 젊은 여자가 흰 옷 아래로 하얀...",NaN,병원,하늘과 바람과 별과 시,윤동주,"불쌍함/연민, 슬픔, 서러움, 안타까움/실망, 힘듦/지침, 패배/자기혐오, 부담/안...","신기함/관심, 불쌍함/연민, 서러움, 슬픔, 안타까움/실망, 당황/난처, 힘듦/지침...","안타까움/실망, 서러움, 힘듦/지침, 고마움, 기대감","서러움, 불안/걱정, 불평/불만, 화남/분노, 의심/불신, 증오/혐오","신기함/관심, 불안/걱정, 불평/불만, 불쌍함/연민"
8,9,8,잃어버렸습니다. 무얼 어디다 잃었는지 몰라 두 손의 호주머니를 더듬어 길에 나...,NaN,길,하늘과 바람과 별과 시,윤동주,"당황/난처, 슬픔, 안타까움/실망, 부끄러움, 부담/안_내킴, 깨달음, 패배/자기혐...","당황/난처, 불안/걱정, 놀람, 서러움, 슬픔, 부끄러움, 감동/감탄, 죄책감, 깨...","안타까움/실망, 서러움, 슬픔, 깨달음, 힘듦/지침","당황/난처, 불안/걱정, 슬픔, 힘듦/지침","불안/걱정, 비장함, 당황/난처, 깨달음"
9,10,9,세상으로부터 돌아오듯이 이제 내 좁은방에 돌아와 불을 끄옵니다. 불을 켜 두는 것은...,NaN,돌아와 보는 밤,하늘과 바람과 별과 시,윤동주,"힘듦/지침, 서러움, 기대감, 깨달음, 슬픔, 비장함","힘듦/지침, 불안/걱정, 부담/안_내킴, 공포/무서움, 불쌍함/연민, 서러움, 슬픔...","힘듦/지침, 깨달음, 편안/쾌적, 슬픔, 불안/걱정","서러움, 슬픔, 힘듦/지침, 불안/걱정","힘듦/지침, 서러움, 깨달음"


In [ ]:
# 특정 시인 작품 데이터
df_poem = df_poem[df_poem['poet'] == '한용운']

In [ ]:
# 특정 시인 라인 데이터
df = df[df["poet"] == "한용운"].copy()

In [ ]:
## 시인 전체 합산 데이터(행+작품)
df_merged = pd.concat([df, df_poem], axis=0, ignore_index=True)

In [ ]:
text_list = df_merged["text"].tolist()

In [ ]:
# 기초 세팅
REPO_ID = "AKS-DHLAB/KPoEM" # 허깅페이스에 업로드된 감정분류모델 id
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") #GPU 사용
THRESH_HOLD = 0.3

In [ ]:
class KPoEM_Classifier(nn.Module):
    def __init__(self, repo_id, device):
        self.labels = [
            '불평/불만', '환영/호의', '감동/감탄', '지긋지긋', '고마움', '슬픔', '화남/분노', '존경',
            '기대감', '우쭐댐/무시함', '안타까움/실망', '비장함', '의심/불신', '뿌듯함', '편안/쾌적',
            '신기함/관심', '아껴주는', '부끄러움', '공포/무서움', '절망', '한심함', '역겨움/징그러움',
            '짜증', '어이없음', '없음', '패배/자기혐오', '귀찮음', '힘듦/지침', '즐거움/신남', '깨달음',
            '죄책감', '증오/혐오', '흐뭇함(귀여움/예쁨)', '당황/난처', '경악', '부담/안_내킴', '서러움',
            '재미없음', '불쌍함/연민', '놀람', '행복', '불안/걱정', '기쁨', '안심/신뢰'
        ]
        num_labels = len(self.labels)
        #모델 & 토크나이저 로드
        super().__init__()
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(repo_id)
        self.electra = AutoModel.from_pretrained(repo_id)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.1),
            nn.Linear(self.electra.config.hidden_size, num_labels)
        )

        weights_path = hf_hub_download(repo_id=repo_id, filename="classifier_state.bin") #가중치 불러오기
        self.classifier.load_state_dict(torch.load(weights_path, map_location=self.device))
        self.to(self.device)
        self.eval()

    # 텍스트를 입력받아 최종 logits 반환
    def forward(self, text: str):
        encoding = self.tokenizer(
          text,
          add_special_tokens=True,
          max_length=512,
          padding="max_length",
          truncation=True,
          return_tensors='pt',
        ).to(self.device)

        model_inputs = {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
        }
        if "token_type_ids" in encoding:
            model_inputs["token_type_ids"] = encoding["token_type_ids"]

        with torch.no_grad():
            outputs = self.electra(
                **model_inputs
            )

        pooled_output = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(pooled_output)
        return logits

    def analyze(self, text: str, threshold=0):
        logits = self.forward(text)
        probabilities = torch.sigmoid(logits.squeeze()) #확률로 변환 → threshold 이상이면 선택
        predictions = (probabilities > threshold).int()

        result_dict = {
            self.labels[i]: float(round(probabilities[i].item(), 3))
            for i, label_id in enumerate(predictions)
            if label_id == 1
        }

        # 확률값 기준 내림차순 정렬된 dict로 반환
        result_dict = dict(sorted(result_dict.items(), key=lambda x: x[1], reverse=True))
        return result_dict

In [ ]:
# KPoEM 모델 로드
print(f"... '{DEVICE}' 환경에서 '{REPO_ID}' 모델을 로드하고 있습니다 ...")
kpoem_model = KPoEM_Classifier(repo_id=REPO_ID, device=DEVICE)
print("KPoEM 모델을 성공적으로 로드하였습니다.")

... 'cuda' 환경에서 'AKS-DHLAB/KPoEM' 모델을 로드하고 있습니다 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

KPoEM 모델을 성공적으로 로드하였습니다.


In [ ]:
import numpy as np

def shap_predict(texts): #SHAP Explainer가 호출하는 예측 함수.
    # 모델을 평가 모드로 전환
    # Dropout 등이 비활성화되어 예측 결과가 고정됨
    kpoem_model.eval()
    # 각 텍스트의 예측 확률을 저장할 리스트
    outputs = []
    # SHAP는 모델을 매우 여러 번 호출하므로,
    # gradient 계산을 끄면 메모리 사용량과 계산 비용을 줄일 수 있음
    with torch.no_grad():
     # SHAP가 넘겨준 텍스트들을 하나씩 처리
        for text in texts:
            logits = kpoem_model.forward(text)              # (1, num_labels)
            # multi-label classification이므로 sigmoid 사용
            # 각 감정별 독립 확률값으로 변환
            probs = torch.sigmoid(logits)             # (1, num_labels)

            outputs.append(
                probs.squeeze().detach().cpu().numpy()
            ) # squeeze()로 (1, 44)를 (44,) 형태로 변경

    return np.vstack(outputs)


In [ ]:
import shap

explainer = shap.Explainer(
    shap_predict,
    masker=shap.maskers.Text(kpoem_model.tokenizer)
)


# 시인들

In [ ]:
texts = (
    df_merged["text"]
    .dropna()
    .astype(str)
    .map(lambda s: s.strip())
)

texts = [t for t in texts if t]  # 빈 문자열 제거
print("texts count:", len(texts))


texts count: 1336


In [ ]:
df_merged.head()

,line_id,poem_id,text,sub_title,title,poet,annotator_01,annotator_02,annotator_03,annotator_04,annotator_05,seg_id,poetry_book
0,2822.0,194,"임은 갔습니다. 아아, 사랑하는 나의 임은 갔습니다.",NaN,님의 침묵,한용운,"슬픔, 서러움, 안타까움/실망","슬픔, 서러움, 안타까움/실망, 불안/걱정, 절망","경악, 당황/난처, 부담/안_내킴, 불쌍함/연민, 비장함, 서러움, 슬픔, 안타까움/실망","당황/난처, 안타까움/실망, 슬픔, 서러움, 경악, 아껴주는","불안/걱정, 슬픔, 서러움, 절망, 힘듦/지침",NaN,NaN
1,2823.0,194,푸른 산빛을 깨치고 단풍나무 숲을 향하여 난 작은 길을 걸어서 차마 떨치고 갔습니다.,NaN,님의 침묵,한용운,"비장함, 슬픔, 당황/난처","비장함, 슬픔, 서러움, 불안/걱정, 부담/안_내킴, 안타까움/실망, 패배/자기혐오","부담/안_내킴, 불평/불만, 비장함, 서러움, 안타까움/실망","비장함, 뿌듯함, 불안/걱정","비장함, 힘듦/지침",NaN,NaN
2,2824.0,194,황금의 꽃같이 굳고 빛나던 옛 맹세는 차디찬 티끌이 되어서 한숨의 미풍에 날아갔습니다.,NaN,님의 침묵,한용운,"비장함, 슬픔, 어이없음, 서러움, 안타까움/실망","비장함, 서러움, 슬픔, 안타까움/실망, 절망, 부끄러움","당황/난처, 불쌍함/연민, 부담/안_내킴, 안타까움/실망","슬픔, 안타까움/실망, 의심/불신, 절망, 힘듦/지침","서러움, 안타까움/실망, 패배/자기혐오",NaN,NaN
3,2825.0,194,날카로운 첫키스의 추억은 나의 운명의 지침을 돌려 놓고 뒷걸음쳐서 사라졌습니다.,NaN,님의 침묵,한용운,"안타까움/실망, 불안/걱정, 깨달음","깨달음, 놀람, 비장함, 서러움, 기쁨, 슬픔","기쁨, 깨달음, 비장함, 안타까움/실망","놀람, 당황/난처, 슬픔, 안타까움/실망","기쁨, 놀람, 슬픔, 서러움",NaN,NaN
4,2826.0,194,나는 향기로운 임의 말소리에 귀먹고 꽃다운 임의 얼굴에 눈멀었습니다.,NaN,님의 침묵,한용운,"아껴주는, 흐뭇함(귀여움/예쁨), 환영/호의","아껴주는, 흐뭇함(귀여움/예쁨), 감동/감탄, 행복","기대감, 기쁨, 감동/감탄, 아껴주는, 존경, 행복","깨달음, 비장함, 슬픔, 행복, 흐뭇함(귀여움/예쁨), 아껴주는","당황/난처, 슬픔, 증오/혐오",NaN,NaN


# shap 분석 + 디버깅

In [ ]:
import os, math, json, pickle
from typing import Any, Dict, Sequence, List

In [ ]:
def _safe_explain_batch(explainer, batch: List[str]):
    """
    배치가 에러나면 이진 분할로 문제 입력을 찾아낸다.
    반환: (ok_explanations_list, bad_texts_list)
    ok_explanations_list: shap.Explanation들을 나중에 합칠 수 있음
    """
    try:
        sv = explainer(batch)
        return [sv], []  # 통째로 성공
    except Exception as e:
        # 배치가 1개인데도 터지면 이 문장이 문제
        if len(batch) == 1:
            return [], [{"text": batch[0], "error": repr(e)}]

        mid = len(batch) // 2
        left_ok, left_bad = _safe_explain_batch(explainer, batch[:mid])
        right_ok, right_bad = _safe_explain_batch(explainer, batch[mid:])
        return left_ok + right_ok, left_bad + right_bad
        # 배치 길이가 2 이상이면 → 반으로 나누어 재귀 호출


In [ ]:
def shap_batched_run_robust(
    explainer,
    texts: Sequence[str],
    batch_size: int = 16,
    out_dir: str = "shap_chunks",
    prefix: str = "dongju",
    start_batch: int = 120, # batch 시작
    verbose: bool = True,
    bad_log_path: str = None,
) -> Dict[str, Any]:

    os.makedirs(out_dir, exist_ok=True)
    if bad_log_path is None:
        bad_log_path = os.path.join(out_dir, f"{prefix}_bad_texts.jsonl")

    total = len(texts)
    n_batches = math.ceil(total / batch_size)

    saved = 0
    total_bad = 0

    for b_idx in range(n_batches):
        if b_idx < start_batch:
            continue

        batch = texts[b_idx * batch_size : (b_idx + 1) * batch_size]
        batch = [x for x in batch if isinstance(x, str) and x.strip()]  # 빈문장 제거
        if not batch:
            if verbose:
                print(f"[skip] empty batch at {b_idx}")
            continue

        ok_list, bad_list = _safe_explain_batch(explainer, batch) #에러나면 디버깅하는 함수

        # 문제 문장 로그 남기기
        if bad_list:
            total_bad += len(bad_list)
            with open(bad_log_path, "a", encoding="utf-8") as f:
                for row in bad_list:
                    row.update({"batch_index": b_idx})
                    f.write(json.dumps(row, ensure_ascii=False) + "\n")
            if verbose:
                print(f"[warn] batch {b_idx}: bad_texts={len(bad_list)} logged -> {bad_log_path}")

        # 정상 결과 저장 (배치가 쪼개져 여러 Explanation이 나올 수 있음)
        for part_i, sv in enumerate(ok_list):
            out_path = os.path.join(out_dir, f"{prefix}_shap_batch{b_idx:04d}_part{part_i:02d}.pkl")
            with open(out_path, "wb") as f:
                pickle.dump(
                    {
                        "batch_index": b_idx,
                        "part_index": part_i,
                        "batch_size": len(getattr(sv, "data", [])),
                        "start_row": b_idx * batch_size,
                        "end_row": b_idx * batch_size + len(batch),
                        "texts": batch,          # 원 배치 텍스트(참고용)
                        "shap_values": sv,
                    },
                    f,
                    protocol=pickle.HIGHEST_PROTOCOL,
                )
            saved += 1

        if verbose:
            print(f"[{b_idx+1}/{n_batches}] saved_parts={len(ok_list)} (bad={len(bad_list)})")

    return {
        "total_texts": total,
        "batch_size": batch_size,
        "n_batches": n_batches,
        "saved_files": saved,
        "bad_texts": total_bad,
        "out_dir": out_dir,
        "prefix": prefix,
        "bad_log_path": bad_log_path,
    }


In [ ]:
meta = shap_batched_run_robust(
    explainer=explainer,
    texts=texts,
    batch_size=8,
    out_dir="shap_han",
    prefix="yongun",
    start_batch=122,
    verbose=True,
)
print(meta)

PartitionExplainer explainer:  12%|█▎        | 1/8 [00:00<?, ?it/s]

  0%|          | 0/90 [00:00<?, ?it/s]

PartitionExplainer explainer:  62%|██████▎   | 5/8 [00:31<00:06,  2.21s/it]

  0%|          | 0/272 [00:00<?, ?it/s]

PartitionExplainer explainer: 9it [00:54,  6.87s/it]


...

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 100%|██████████| 8/8 [01:55<00:00, 15.67s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 9it [02:12, 16.60s/it]

[167/167] saved_parts=1 (bad=0)
{'total_texts': 1336, 'batch_size': 8, 'n_batches': 167, 'saved_files': 45, 'bad_texts': 0, 'out_dir': 'shap_han', 'prefix': 'yongun', 'bad_log_path': 'shap_han/yongun_bad_texts.jsonl'}


In [ ]:
!zip -r shap_han.zip shap_han

updating: shap_han/ (stored 0%)
updating: shap_han/yongun_shap_batch0069_part00.pkl (deflated 53%)
updating: shap_han/yongun_bad_texts.jsonl (deflated 45%)
updating: shap_han/yongun_shap_batch0062_part00.pkl (deflated 56%)
updating: shap_han/yongun_shap_batch0056_part00.pkl (deflated 52%)
updating: shap_han/yongun_shap_batch0053_part00.pkl (deflated 53%)
updating: shap_han/yongun_shap_batch0064_part00.pkl (deflated 51%)
updating: shap_han/yongun_shap_batch0058_part00.pkl (deflated 55%)
updating: shap_han/yongun_shap_batch0073_part00.pkl (deflated 53%)
updating: shap_han/yongun_shap_batch0074_part00.pkl (deflated 56%)
updating: shap_han/yongun_shap_batch0076_part00.pkl (deflated 53%)
updating: shap_han/yongun_shap_batch0065_part00.pkl (deflated 55%)
updating: shap_han/yongun_shap_batch0055_part00.pkl (deflated 55%)
updating: shap_han/yongun_shap_batch0060_part00.pkl (deflated 56%)
updating: shap_han/yongun_shap_batch0079_part00.pkl (deflated 59%)
updating: shap_han/yongun_shap_batch0067